In [1]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))


True
Tesla T4


In [6]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [7]:
import os
print(os.listdir("/content/drive/MyDrive")[:50])


['Colab Notebooks', 'Getting started.pdf', 'FORM 2.docx', 'Classroom', 'Copy of F 2 - RK  - w 09.docx', 'ICT', 'IMG-20210619-WA0009.jpg', 'IMG-20220316-WA0093.jpg', 'IMG-20220316-WA0092.jpg', '5AC43C12-2AFA-4663-B8EC-D00B2008563C.pdf', 'Screenshot_20220801-200014_Drive.jpg', 'WIN_20220802_20_57_30_Pro.mp4', 'WIN_20220803_21_11_09_Pro (1).mp4', 'WIN_20220803_21_11_09_Pro.mp4', 'MCBackup - My Contacts', 'Apiit', 'WDOS ASSI (2).zip', '101032992971500_20240117063906122.pdf', 'The P4 Language Consortium Gsoc Proposal 2024.docx', 'Untitled document.gdoc', 'APIIT edit', 'jetbrains.zip', 'Dinuka Jayasundara - Cv.pdf', 'CcArtisan.zip', 'Dinuka Jayasundara (1).pdf', 'Dinuka Jayasundara.pdf', 'receipt.pdf', 'Sher', 'Receipt234874020250504015658567000 (1).pdf', 'Receipt234874020250504015658567000.pdf', 'WhatsApp Video 2025-07-19 at 23.44.04_9ff0d7fb.mp4', 'Receipt234874020250901175306792000 (2).pdf', 'Receipt234874020250901175306792000 (1).pdf', 'Receipt234874020250901175306792000.pdf', 'MIND MAP'

In [9]:
from pathlib import Path

drive_root = Path("/content/drive/MyDrive")

matches = list(drive_root.rglob("archive1.zip"))
print("Found:", len(matches))
for m in matches[:20]:
    print(m)


Found: 1
/content/drive/MyDrive/Vehicle_Damage_Detection/data/processed/archive1.zip


In [10]:
from pathlib import Path

DATA_DIR = Path("/content/drive/MyDrive/Vehicle_Damage_Detection/data/processed/archive1")
print("DATA_DIR exists:", DATA_DIR.exists())
print("train exists:", (DATA_DIR/"train").exists())
print("val exists:", (DATA_DIR/"val").exists())


DATA_DIR exists: True
train exists: True
val exists: True


In [11]:
from pathlib import Path

DATA_DIR = Path("/content/drive/MyDrive/Vehicle_Damage_Detection/data/processed/archive1")

train_damage = DATA_DIR / "train" / "00-damage"
train_whole  = DATA_DIR / "train" / "01-whole"
val_damage   = DATA_DIR / "val" / "00-damage"
val_whole    = DATA_DIR / "val" / "01-whole"

def count_imgs(p):
    return len(list(p.glob("*.jpg"))) + len(list(p.glob("*.jpeg"))) + len(list(p.glob("*.png")))

print("Train damage:", count_imgs(train_damage))
print("Train whole :", count_imgs(train_whole))
print("Val damage  :", count_imgs(val_damage))
print("Val whole   :", count_imgs(val_whole))


Train damage: 54
Train whole : 792
Val damage  : 14
Val whole   : 199


In [12]:
!pip -q install ultralytics


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 75.5 MB/s eta 0:00:00


In [13]:
from ultralytics import YOLO

DATA_DIR = "/content/drive/MyDrive/Vehicle_Damage_Detection/data/processed/archive1"

model = YOLO("yolo11n-cls.pt")   # small + fast (good start)

results = model.train(
    data=DATA_DIR,
    imgsz=224,
    epochs=10,
    batch=32
)


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.12 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/Vehicle_Damage_Detection/data/processed/archive1, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=10, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0

In [14]:
from pathlib import Path
import shutil

src = Path("/content/runs/classify")
dst = Path("/content/drive/MyDrive/Vehicle_Damage_Detection/runs/classify")

dst.parent.mkdir(parents=True, exist_ok=True)

if dst.exists():
    shutil.rmtree(dst)

shutil.copytree(src, dst)
print(" Copied results to:", dst)


 Copied results to: /content/drive/MyDrive/Vehicle_Damage_Detection/runs/classify


In [15]:
from pathlib import Path
DATA_DIR = Path("/content/drive/MyDrive/Vehicle_Damage_Detection/data/processed/archive1")
print("Train:", sum(1 for _ in DATA_DIR.rglob("train/*/*")))
print("Val  :", sum(1 for _ in DATA_DIR.rglob("val/*/*")))


Train: 1840
Val  : 460


In [18]:
from pathlib import Path

runs = Path("/content/runs/classify")

best_list = list(runs.rglob("best.pt"))
print("Found best.pt:", len(best_list))

for p in best_list[:20]:
    print(p)


Found best.pt: 1
/content/runs/classify/train/weights/best.pt


In [19]:
from ultralytics import YOLO
from pathlib import Path

runs = Path("/content/runs/classify")
best_path = max(runs.rglob("best.pt"), key=lambda p: p.stat().st_mtime)  # newest best.pt

print("Using model:", best_path)

model = YOLO(str(best_path))


Using model: /content/runs/classify/train/weights/best.pt


In [20]:
source = "/content/drive/MyDrive/Vehicle_Damage_Detection/data/processed/archive1/val/00-damage"

results = model.predict(source=source, imgsz=224, save=True)
print(" Predictions saved!")



image 1/230 /content/drive/MyDrive/Vehicle_Damage_Detection/data/processed/archive1/val/00-damage/0001.JPEG: 224x224 00-damage 0.98, 01-whole 0.02, 4.8ms
image 2/230 /content/drive/MyDrive/Vehicle_Damage_Detection/data/processed/archive1/val/00-damage/0002.JPEG: 224x224 00-damage 1.00, 01-whole 0.00, 4.1ms
image 3/230 /content/drive/MyDrive/Vehicle_Damage_Detection/data/processed/archive1/val/00-damage/0003.JPEG: 224x224 00-damage 0.73, 01-whole 0.27, 8.5ms
image 4/230 /content/drive/MyDrive/Vehicle_Damage_Detection/data/processed/archive1/val/00-damage/0004.JPEG: 224x224 00-damage 1.00, 01-whole 0.00, 4.6ms
image 5/230 /content/drive/MyDrive/Vehicle_Damage_Detection/data/processed/archive1/val/00-damage/0005.JPEG: 224x224 00-damage 0.99, 01-whole 0.01, 4.5ms
image 6/230 /content/drive/MyDrive/Vehicle_Damage_Detection/data/processed/archive1/val/00-damage/0006.JPEG: 224x224 00-damage 0.98, 01-whole 0.02, 3.8ms
image 7/230 /content/drive/MyDrive/Vehicle_Damage_Detection/data/processed/

In [21]:
from pathlib import Path
pred_dirs = sorted(Path("/content/runs/classify").glob("predict*"), key=lambda p: p.stat().st_mtime)
print("Latest predict folder:", pred_dirs[-1])


Latest predict folder: /content/runs/classify/predict


In [22]:
from pathlib import Path
import shutil

src = Path("/content/runs/classify/predict")
dst = Path("/content/drive/MyDrive/Vehicle_Damage_Detection/results/predict_archive1")

dst.parent.mkdir(parents=True, exist_ok=True)

if dst.exists():
    shutil.rmtree(dst)

shutil.copytree(src, dst)
print(" Copied to Drive:", dst)


 Copied to Drive: /content/drive/MyDrive/Vehicle_Damage_Detection/results/predict_archive1


In [1]:
from ultralytics import YOLO
from pathlib import Path

DATA_DIR = Path(r"C:\Users\User\Desktop\Vehicle_Damage_Detection\data\processed\archive1")
print("DATA_DIR exists:", DATA_DIR.exists())

model = YOLO(r"C:\Users\User\Desktop\Vehicle_Damage_Detection\models\base\yolo11n-cls.pt")

results = model.train(
    data=str(DATA_DIR),
    imgsz=224,
    epochs=40,
    batch=32,
    name="archive1_yolo11n_cls"
)

DATA_DIR exists: True
New https://pypi.org/project/ultralytics/8.4.23 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.12  Python-3.11.9 torch-2.10.0+cpu CPU (13th Gen Intel Core i7-13650HX)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\Users\User\Desktop\Vehicle_Damage_Detection\data\processed\archive1, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=40, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=224, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=C:\Users\